In [ ]:
import os
import json
import numpy as np
import librosa
import random

# Configuration
# Primary Path (Absolute Windows Path provided)
AUDIO_PATH_ABSOLUTE = r"C:\Users\maste\OneDrive\Desktop\ASfuera\wetransfer_1-jpg_2022-11-25_1720\Rococo_In_Bloom\songs\Coppélia-Valse_de_la_poupee.mp3"
JSON_PATH_ABSOLUTE = r"C:\Users\maste\OneDrive\Desktop\ASfuera\wetransfer_1-jpg_2022-11-25_1720\Rococo_In_Bloom\songs\Coppélia-Valse_de_la_poupee.json"

# Fallback Path (Relative to project root)
AUDIO_PATH_RELATIVE = os.path.join("songs", "Coppélia-Valse_de_la_poupee.mp3")
JSON_PATH_RELATIVE = os.path.join("songs", "Coppélia-Valse_de_la_poupee.json")

# Select the file that exists
if os.path.exists(AUDIO_PATH_ABSOLUTE):
    AUDIO_FILE = AUDIO_PATH_ABSOLUTE
    OUTPUT_FILE = JSON_PATH_ABSOLUTE
else:
    AUDIO_FILE = AUDIO_PATH_RELATIVE
    OUTPUT_FILE = JSON_PATH_RELATIVE
    
print(f"Target Audio: {AUDIO_FILE}")
print(f"Target Output: {OUTPUT_FILE}")

LANE_COUNT = 5
MIN_GAP_MS = 160  # "Hard-Hard" Density (approx. 6 notes/sec max bursts)
SR = 22050  # Sample Rate
WINDOW_MS = 80  # Analysis window size around each onset
SEED = 123  # For reproducibility

np.random.seed(SEED)
random.seed(SEED)

In [ ]:
# Load Audio & Detect Onsets
if not os.path.exists(AUDIO_FILE):
    raise FileNotFoundError(f"Could not find audio file at: {AUDIO_FILE}. Please check the path.")

y, sr = librosa.load(AUDIO_FILE, sr=SR, mono=True)

onset_times = librosa.onset.onset_detect(y=y, sr=sr, units="time")
sorted_onsets = np.sort(onset_times)

accepted_onsets = []
last_onset = -np.inf
min_gap_s = MIN_GAP_MS / 1000.0
for onset in sorted_onsets:
    if onset - last_onset >= min_gap_s:
        accepted_onsets.append(float(onset))
        last_onset = onset

print(f"Detected onsets: {len(onset_times)} | Accepted onsets: {len(accepted_onsets)}")

In [ ]:
# Calculate Energy for Chords (Loudness detection)
onset_energies = []
window_samples = int((WINDOW_MS / 1000.0) * SR)

for onset in accepted_onsets:
    center = int(onset * sr)
    start = max(0, center - window_samples // 2)
    end = min(len(y), center + window_samples // 2)
    chunk = y[start:end]
    if chunk.size > 0:
        rms = float(np.sqrt(np.mean(chunk**2)))
        onset_energies.append(rms)
    else:
        onset_energies.append(0.0)

# Dynamic Threshold: Top 15% of loudest notes will be double-taps (chords)
if onset_energies:
    energy_threshold = np.percentile(onset_energies, 85)
else:
    energy_threshold = 1.0
    
print(f"Chord Threshold (RMS): {energy_threshold:.4f}")

In [ ]:
# Feature Extraction (Spectral Centroid / Pitch Proxy)
features = []
window_samples = int((WINDOW_MS / 1000.0) * SR)
half_window = max(1, window_samples // 2)

for onset in accepted_onsets:
    center = int(onset * sr)
    start = max(0, center - half_window)
    end = min(len(y), center + half_window)
    window = y[start:end]

    if window.size == 0:
        features.append(0.0)
        continue

    centroid = librosa.feature.spectral_centroid(y=window, sr=sr)
    value = np.nanmean(centroid) if centroid.size > 0 else np.nan

    if np.isnan(value) or np.isinf(value):
        spectrum = np.abs(np.fft.rfft(window))
        freqs = np.fft.rfftfreq(window.size, d=1.0 / sr)
        if spectrum.size > 0:
            peak_idx = int(np.argmax(spectrum))
            value = float(freqs[peak_idx])
        else:
            value = 0.0

    features.append(float(value))

print(f"Extracted features for {len(features)} onsets.")

In [ ]:
# Normalization & Lane Mapping
features_array = np.array(features, dtype=float)

if features_array.size == 0:
    mapped_lanes = []
else:
    # Robust scaling (clamp 5th-95th percentile) to ignore outliers
    p5, p95 = np.percentile(features_array, [5, 95])
    if p5 == p95:
        clamped = np.full_like(features_array, p5)
    else:
        clamped = np.clip(features_array, p5, p95)

    denominator = p95 - p5 if p95 != p5 else 1.0
    normalized = np.clip((clamped - p5) / denominator, 0.0, 1.0)
    mapped_lanes = (normalized * LANE_COUNT).astype(int)
    mapped_lanes = np.clip(mapped_lanes, 0, LANE_COUNT - 1).tolist()

print(f"Lane mapping complete. Sample lanes: {mapped_lanes[:10] if mapped_lanes else 'None'}")

In [ ]:
# Flow Logic (Constraint Solver)
final_lanes = []
prev_lane = None
prev_move = 0  # -1 for left, +1 for right, 0 for none

for lane in mapped_lanes:
    chosen_lane = lane
    if prev_lane is not None and lane == prev_lane:
        preferred_direction = -prev_move if prev_move != 0 else 1
        direction_order = [preferred_direction, -preferred_direction]
        found = False
        for direction in direction_order:
            for offset in range(1, LANE_COUNT):
                candidate = prev_lane + offset * direction
                if 0 <= candidate < LANE_COUNT:
                    chosen_lane = candidate
                    found = True
                    break
            if found:
                break
    move_direction = 0 if prev_lane is None else chosen_lane - prev_lane
    prev_lane = chosen_lane
    prev_move = int(np.sign(move_direction))
    final_lanes.append(int(chosen_lane))

print(f"Applied flow constraints. Sample final lanes: {final_lanes[:10] if final_lanes else 'None'}")

In [ ]:
# Export & Summary (Holds + Chords)
notes = []

for i, (onset, lane, energy) in enumerate(zip(accepted_onsets, final_lanes, onset_energies)):
    time_ms = int(round(onset * 1000))
    
    # 1. Determine Duration (Hold Logic)
    if i < len(accepted_onsets) - 1:
        next_onset = accepted_onsets[i + 1]
        gap_ms = (next_onset * 1000) - time_ms
    else:
        gap_ms = 0

    # Threshold for Hold Notes (> 600ms gap)
    if gap_ms > 600:
        note_type = "hold"
        duration = int(gap_ms - 100) # Leave 100ms breathing room
    else:
        note_type = "normal"
        duration = 0

    # 2. Append Main Note
    notes.append({
        "time": time_ms,
        "lane": int(lane),
        "type": note_type,
        "duration": duration if note_type == "hold" else 0
    })

    # 3. Chord Logic (Double Taps)
    # Only for normal notes (not holds) that are loud (high energy)
    if note_type == "normal" and energy >= energy_threshold:
        chord_lane = (lane + 2) % LANE_COUNT
        if chord_lane == lane:
             chord_lane = (lane + 1) % LANE_COUNT
             
        notes.append({
            "time": time_ms,
            "lane": int(chord_lane),
            "type": "normal"
        })

# Write JSON
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(notes, f, indent=2)

duration_seconds = len(y) / sr if len(y) > 0 else 0.0
notes_per_second = (len(notes) / duration_seconds) if duration_seconds > 0 else 0.0
lane_distribution = np.bincount([n['lane'] for n in notes], minlength=LANE_COUNT)

print(f"Chart exported to: {OUTPUT_FILE}")
print(f"Total Objects: {len(notes)}")
print(f"Notes Per Second: {notes_per_second:.2f}")
print("Lane Distribution:")
for lane_idx, count in enumerate(lane_distribution):
    print(f"  Lane {lane_idx}: {int(count)}")